# NEREID-B — GPU (CuPy) verification on Google Colab

Verifies the CuPy GPU port of `solver.py` on a real CUDA device. The CPU path is the
validated default; the GPU port is designed to be numerically identical (same IEEE-double
kernels; the sparse pressure solve runs on the host in BOTH cases), so a correct port
differs only by floating-point reduction order (~1e-9).

**Before running:** Runtime ▸ Change runtime type ▸ **GPU** (T4 is enough).

Then upload `solver.py` (Files ▸ upload, or the cell below mounts Drive).

In [ ]:
# 1. Confirm the GPU is attached
!nvidia-smi

In [ ]:
# 2. Pin the dependency stack.  CRITICAL: numpy MUST be 1.26.4 — numpy 2.x breaks
#    scipy 1.11.4's sparse import and kills the pressure solver.  cupy-cuda12x matches
#    Colab's CUDA 12 runtime (use cupy-cuda11x if nvidia-smi shows CUDA 11).
!pip -q install numpy==1.26.4 scipy==1.11.4 matplotlib
!pip -q install cupy-cuda12x
# gsw is OPTIONAL (TEOS-10 eos_mode); it declares numpy>=2 but works on 1.26.4.
# Install it ONLY if you need eos_mode="teos10"; otherwise skip to keep numpy pinned.
# !pip -q install --no-deps gsw
print('IMPORTANT: Runtime ▸ Restart session now, then run the cells below (do NOT re-run this cell).')

In [ ]:
# 3. (after restart) sanity-check the versions
import numpy, scipy, cupy
print('numpy', numpy.__version__, '(must be 1.26.4)')
print('scipy', scipy.__version__)
print('cupy ', cupy.__version__)
print('CUDA devices', cupy.cuda.runtime.getDeviceCount())

In [ ]:
# 4. Make sure solver.py is here (upload via the Files panel, or clone your repo).
import os; assert os.path.exists('solver.py'), 'Upload solver.py to the Colab working dir first.'
print('solver.py found:', os.path.abspath('solver.py'))

In [ ]:
# 5. Device status
!python solver.py --gpu-check

In [ ]:
# 6. THE verification: same short sim on CPU and GPU, report max|CPU-GPU|.
#    PASS = the CuPy port reproduces the validated CPU result to float round-off.
!python solver.py --gpu-verify

In [ ]:
# 7. Confirm the invariants still hold on this machine (CPU backend), then
#    run a real prediction on the GPU.
!python solver.py --selftest
!python solver.py --gpu --nx 64 --ny 40 --nz 28 --t_end 360

## Interpreting the result

- `--gpu-verify` **VERIFIED** (rel diffs ~1e-9 to 1e-6 per field) → the CuPy port is correct;
  the GPU path can now be used for the large/fine-grid runs the CPU can't afford.
- A **MISMATCH** (rel diff ≫ 1e-6) points at a host↔device boundary that still leaks a
  CuPy array into a host (SciPy/`np.add.at`) kernel — report the field name that failed.
- The single host↔device round-trip per step is the sparse pressure solve (kept on the CPU
  LU by design); for very large grids that transfer is the next thing to profile.